# Analyse all possible VNC innervating Neurons
Analyse all the data from the [fauai-13 notebook](notebooks/fauai-13_DetailedCotransmissionPatterns.ipynb)

## Libraries and Base dataframes

In [ ]:
# libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Get list of all csv files with `new_nt_results_*.csv`
# csv_files = [f for f in os.listdir() if f.startswith('new_nt_results_') and f.endswith('.csv')]
# print(csv_files)
cell_type_csv_files = [f for f in os.listdir("./fauai-13_data/cell_type") if f.endswith('.csv')]
cell_class_csv_files = [f for f in os.listdir("./fauai-13_data/cell_class") if f.endswith('.csv')]
cell_flow_csv_files = [f for f in os.listdir("./fauai-13_data/cell_flow") if f.endswith('.csv')]
    
# Now add the directory to create the path
cell_type_csv_files = [f"./fauai-13_data/cell_type/{f}" for f in cell_type_csv_files]
cell_class_csv_files = [f"./fauai-13_data/cell_class/{f}" for f in cell_class_csv_files]
cell_flow_csv_files = [f"./fauai-13_data/cell_flow/{f}" for f in cell_flow_csv_files]
print(cell_type_csv_files)
print(cell_class_csv_files)
print(cell_flow_csv_files)

In [ ]:
cell_type_df = pd.concat([pd.read_csv(f) for f in cell_type_csv_files], ignore_index=True)
cell_class_df = pd.concat([pd.read_csv(f) for f in cell_class_csv_files], ignore_index=True)
cell_flow_df = pd.concat([pd.read_csv(f) for f in cell_flow_csv_files], ignore_index=True)
display(cell_type_df.head())
display(cell_class_df.head())
display(cell_flow_df.head())

nt_results_df = pd.concat([cell_type_df, cell_class_df, cell_flow_df], axis=0)
display(nt_results_df.head())

display(nt_results_df.describe())

## Overall Statistics

### Descriptive Statistics

In [ ]:
display(nt_results_df.describe())
print(f"Number of rows in the DataFrame: {nt_results_df.shape[0]}")

# Identify quantitative columns
# quant_cols = nt_results_df.select_dtypes(include=['float64', 'int64']).columns.tolist()
quant_cols = ["num_synapses", "GABA_ratio", "ACh_ratio", "Glut_ratio", "Oct_ratio", "Ser_ratio", "DA_ratio", "Unknown_ratio"]

# Plot data spread
fig, ax = plt.subplots(len(quant_cols), 1, figsize=(12, 6 * len(quant_cols)))
for i, col in enumerate(quant_cols):
    print(col)
    sns.histplot(data=nt_results_df, x=col, bins=25, kde=True, ax=ax[i])
    ax[i].set_title(f'Distribution of {col}')


# Plot data spread after removing duplicates
keep_cols = ["root_id"] + quant_cols
temp_df = nt_results_df[keep_cols].drop_duplicates()

n_plots = len(quant_cols) - 1
n_rows = 2
n_cols = 4
nr = 0
nc = 0

fig, ax = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
for i, col in enumerate(quant_cols):
    sns.histplot(data=temp_df, x=col, bins=25, kde=True, ax=ax[nr, nc])
    ax[nr, nc].set_title(f'Distribution of {col}')
    if col != "num_synapses":
        ax[nr, nc].set_xlim(0, 1)
    nc += 1
    if nc >= n_cols:
        nc = 0
        nr += 1
    
            
display(temp_df[quant_cols].describe())

In [ ]:
## Get list of thresholds
ratio_thresholds = [1/6, 1/5, 1/4, 1/3, 1/2] #NB These are added as a customiseable feature here
softmax_thresholds = nt_results_df["softmax_threshold"].unique().tolist()
bandwidth_quantile_values = nt_results_df["bandwidth_quantile_value"].unique().tolist()
min_synapses_ratio_values = nt_results_df["min_synapses_ratio_value"].unique().tolist()

#print to confirm
print(f"Ratio thresholds: {ratio_thresholds}")
print(f"Softmax thresholds: {softmax_thresholds}")
print(f"Bandwidth quantile values: {bandwidth_quantile_values}")
print(f"Min synapses ratio values: {min_synapses_ratio_values}")


In [ ]:
original_nt_results_df = nt_results_df.copy()

### Add in the ratio threshold predictions

In [ ]:
NT_ratio_columns = ["GABA_ratio", "ACh_ratio", "Glut_ratio", "Oct_ratio", "Ser_ratio", "DA_ratio", "Unknown_ratio"]
for i, ratio in enumerate(ratio_thresholds):
    print(f"Ratio threshold {i}: {ratio}")
    if i == 0:
        # Add ratio threshold column to DataFrame
        nt_results_df["ratio_threshold"] = ratio
        # Now add boolean columns for each NT if the NT_ratio >= ratio
        for col in NT_ratio_columns:
            nt_results_df[f"{col}_above"] = nt_results_df[col] >= ratio
    else:
        _df = original_nt_results_df.copy()
        _df["ratio_threshold"] = ratio
        for col in NT_ratio_columns:
            _df[f"{col}_above"] = _df[col] >= ratio
        nt_results_df = pd.concat([nt_results_df, _df], ignore_index=True)
        print(f"adding {_df.shape[0]} rows, length is now {nt_results_df.shape[0]}")

display(nt_results_df)

In [ ]:
nts = ["GABA", "ACh", "Glut", "Oct", "Ser", "DA", "Unknown"]
# Now create a joint prediction between the cluster and ratio predictions
new_cols = []
for nt in nts:
    col_name = f"{nt}_joint_prediction"
    nt_results_df[col_name] = nt_results_df[f"{nt}_ratio_above"] | nt_results_df[f"{nt}_clusters"]
    new_cols.append(col_name)

nt_results_df["num_nts"] = nt_results_df[new_cols].sum(axis=1)
nt_results_df["is_cotransmitting"] = nt_results_df["num_nts"] > 1
nt_results_df.head()

## Determine the best threshold settings

In [ ]:
# REMINDER OF THRESHOLDS
# ratio_thresholds = [1/6, 1/5, 1/4, 1/3, 1/2] #NB These are added as a customiseable feature here
# softmax_thresholds = nt_results_df["softmax_threshold"].unique().tolist() #This is just 0.25 can ignore
# bandwidth_quantile_values = nt_results_df["bandwidth_quantile_value"].unique().tolist()
# min_synapses_ratio_values = nt_results_df["min_synapses_ratio_value"].unique().tolist()

# n_rows = len(ratio_thresholds) * len(bandwidth_quantile_values) * len(min_synapses_ratio_values)
nts = ["GABA", "ACh", "Glut", "Oct", "Ser", "DA", "Unknown"]

for ratio in ratio_thresholds:
    for bandwidth in bandwidth_quantile_values:
        for min_synapses in min_synapses_ratio_values:
            fig, axs = plt.subplots(2, 2, figsize=(10, 10))
            _df = nt_results_df[
                (nt_results_df["ratio_threshold"] == ratio) &
                (nt_results_df["bandwidth_quantile_value"] == bandwidth) &
                (nt_results_df["min_synapses_ratio_value"] == min_synapses)
            ]
            # Perform absolute count plots
            sns.countplot(data=_df, x="num_nts", ax=axs[0,0])
            axs[0,0].set_xlim(-0.5, 7)
            axs[0,0].set_title('Absolute Count of Cells by Number of NTs')
            axs[0,0].set_xlabel('Number of NTs')
            axs[0,0].set_ylabel('Count')

            # Perform relative count plots
            n_bins = _df["num_nts"].nunique()
            sns.histplot(data=_df, x="num_nts", ax=axs[0,1], stat="percent", kde=True, bins=n_bins)
            axs[0,1].set_xlim(-0.5, 7)
            axs[0,1].set_title('Relative Count of Cells by Number of NTs')
            axs[0,1].set_xlabel('Number of NTs')
            axs[0,1].set_ylabel('Percentage')

            # Collate number of each transmitter
            for i, nt in enumerate(nts):
                nt_col = f"{nt}_joint_prediction"
                axs[1,0].barh(i, (_df[nt_col].sum()), label=nt)
                axs[1,0].set_title('Number of Cells Expressing Each NT')
                axs[1,0].set_xlabel('Number')
                axs[1,0].legend(title='Neurotransmitter')
            axs[1,0].set_yticks(range(len(nts)))
            axs[1,0].set_yticklabels(nts)

            # Collate number of each transmitter (percentages)
            for i, nt in enumerate(nts):
                nt_col = f"{nt}_joint_prediction"
                axs[1,1].barh(i, (_df[nt_col].sum() / _df[nt_col].shape[0]) * 100, label=nt)
                axs[1,1].set_title('Percentage of Cells Expressing Each NT')
                axs[1,1].set_xlabel('Percentage')
                # axs[1,1].legend(title='Neurotransmitter')
            axs[1,1].set_yticks(range(len(nts)))
            axs[1,1].set_yticklabels(nts)

            fig.suptitle(f"Ratio: {ratio}, Bandwidth: {bandwidth}, Min Synapses: {min_synapses}")
            plt.tight_layout()
            # Save figure as png
            fig_name = f"./fauai-16_data/NTcounts_ratio_{ratio}_bandwidth_{bandwidth}_min_synapses_{min_synapses}.png"
            plt.savefig(fig_name)

Think the previous settings were still reasonable
NT_ratios
* Ratio: `0.2` (i.e. at least 20%)

Mean-Shift Clustering Thresholds
* Bandwidth: `0.01`
* Min Synapses: `0.05` (I.e. a cluster needs to be at least 5% of the synapses of the neuron)

In [ ]:
### Add in the ratio threshold predictions
# Ratio thresholds: [0.16666666666666666, 0.2, 0.25, 0.3333333333333333, 0.5]
# Softmax thresholds: [0.25]
# Bandwidth quantile values: [0.1428571428571428, 0.04, 0.02, 0.01]
# Min synapses ratio values: [0.01, 0.05, 0.1]


# Just produce plots so we can see what happens
assessed_ratios = [0.16666666666666666, 0.2, 0.5]
cluster_thresholds = [(0.1428571428571428, 0.01), (0.02, 0.01), (0.01, 0.05),(0.01, 0.1)]

nts = ["GABA", "ACh", "Glut", "Oct", "Ser", "DA", "Unknown"]

fig, axs = plt.subplots(3, 4, figsize=(20, 10), sharex=True, sharey=True)
for r, ratio in enumerate(assessed_ratios):
    for c, (bandwidth, min_synapses) in enumerate(cluster_thresholds):
        _df = nt_results_df[
            (nt_results_df["ratio_threshold"] == ratio) &
            (nt_results_df["bandwidth_quantile_value"] == bandwidth) &
            (nt_results_df["min_synapses_ratio_value"] == min_synapses)
        ]

        # Perform relative count plots
        n_bins = _df["num_nts"].nunique()
        sns.histplot(data=_df, x="num_nts", ax=axs[r,c], stat="percent", kde=True, bins=n_bins)
        axs[r,c].set_xlim(-0.5, 7)
        axs[r,c].set_ylim(0, 100)
        # axs[r,c].set_title('Relative Count of Cells by Number of NTs')
        axs[r,c].set_xlabel('Number of NTs')
        axs[r,c].set_ylabel('Percentage')

    plt.tight_layout()


fig, axs = plt.subplots(3, 4, figsize=(20, 10), sharex=True, sharey=True)
for r, ratio in enumerate(assessed_ratios):
    for c, (bandwidth, min_synapses) in enumerate(cluster_thresholds):
        _df = nt_results_df[
            (nt_results_df["ratio_threshold"] == ratio) &
            (nt_results_df["bandwidth_quantile_value"] == bandwidth) &
            (nt_results_df["min_synapses_ratio_value"] == min_synapses)
        ]
        # Collate number of each transmitter (percentages)
        for i, nt in enumerate(nts):
            nt_col = f"{nt}_joint_prediction"
            axs[r,c].barh(i, (_df[nt_col].sum() / _df[nt_col].shape[0]) * 100, label=nt)
            # axs[r,c].set_title('Percentage of Cells Expressing Each NT')
            # axs[r,c].set_xlabel('Percentage')
            # axs[r,c].legend(title='Neurotransmitter')
        axs[r,c].set_yticks(range(len(nts)))
        axs[r,c].set_yticklabels(nts)

plt.tight_layout()

In [ ]:
chosen_nt_results_df = nt_results_df[
                (nt_results_df["ratio_threshold"] == 0.2) &
                (nt_results_df["bandwidth_quantile_value"] == 0.01) &
                (nt_results_df["min_synapses_ratio_value"] == 0.05)
            ]

## View combinations

### Get all possible combinations

In [ ]:
# Now work out which combinations of NTs are most common for the low bandwidth group
nts = ["GABA", "ACh", "Glut", "Oct", "Ser", "DA", "Unknown"]

# Work out all the possible combinations
from itertools import product

# create all boolean combinations (True/False)
combos = list(product([True, False], repeat=len(nts)))
combos_df = pd.DataFrame(combos, columns=nts)
combos_df['num_nts'] = combos_df.sum(axis=1)
combos_df.sort_values('num_nts', inplace=True)
combos_df.reset_index(inplace=True, drop=True)
display(combos_df)

### Count the occurences of each combination for my chosen threshold settings

In [ ]:
combination_data = []
# Now count the occurrences of each combination in the low bandwidth filtered DataFrame
for tot_nts in range(1, len(nts) + 1):
    subset_combos_df = combos_df[combos_df['num_nts'] == tot_nts]
    for i in range(subset_combos_df.shape[0]):
        combo = subset_combos_df.iloc[i]
        # Find the nts in the combo
        nts_in_combo = [nt for nt in nts if combo[nt]]
        combo_nt = combo[nts]
        joint_nt_cols = [f"{nt}_joint_prediction" for nt in nts]
        _df = chosen_nt_results_df[
            (chosen_nt_results_df[joint_nt_cols[0]] == combo_nt[nts[0]]) &
            (chosen_nt_results_df[joint_nt_cols[1]] == combo_nt[nts[1]]) &
            (chosen_nt_results_df[joint_nt_cols[2]] == combo_nt[nts[2]]) &
            (chosen_nt_results_df[joint_nt_cols[3]] == combo_nt[nts[3]]) &
            (chosen_nt_results_df[joint_nt_cols[4]] == combo_nt[nts[4]]) &
            (chosen_nt_results_df[joint_nt_cols[5]] == combo_nt[nts[5]])
             ]        
        #Add to key_data dict
        key_data = {
            'combination_index': i + 1,
            'num_nts': tot_nts,
            'nts_in_combo': nts_in_combo,
            'num_neurons': _df.shape[0]
        }
        combination_data.append(key_data)

combination_data_df = pd.DataFrame(combination_data)
display(combination_data_df.head(15))

In [ ]:
# Now plot the combinations 
for i in range(1, 8):
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    _df = combination_data_df[combination_data_df['num_nts'] == i]
    # drop _df rows where 'num_neurons' is 0
    _df = _df[_df['num_neurons'] > 0]
    for j, row in _df.iterrows():
        x_name = str(row['nts_in_combo'])
        ax.bar(x_name, row['num_neurons'], label=x_name)
        # Rotate the x-ticks
        ax.tick_params(axis='x', rotation=45)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment('right')
    ax.set_title(f'Number of NTs: {i}')
    display(_df.sort_values('num_neurons', ascending=False))

## Explore Classification and Cell Cell Type combinations

In [ ]:
classifications = chosen_nt_results_df['classification_system'].unique().tolist()
cell_types = chosen_nt_results_df['cell_type'].unique().tolist()
print(classifications)
print(cell_types)

### Plot a heatmap of everything

In [ ]:
nts = ["GABA", "ACh", "Glut", "Oct", "Ser", "DA", "Unknown"]
all_cell_type_data = []
joint_nt_cols = [f"{nt}_joint_prediction" for nt in nts]

for classification in classifications: #for every classification type
    for cell_type in cell_types: # for cell_type in cell_types
        filter_name = f"{classification}_{cell_type}"
        print(filter_name)
        cell_type_combos = {}
        _df = chosen_nt_results_df[(chosen_nt_results_df['cell_type'] == cell_type) & (chosen_nt_results_df['classification_system'] == classification)]

        n_nts = []
        nt_present = {nt: 0 for nt in nts}

        if _df.shape[0] == 0: # no cells of this type, skip
            continue

        for j in range(_df.shape[0]):
            row = _df.iloc[j]
            row_nts = row[joint_nt_cols]
            # rename row_nts to just nt
            row_nts = row_nts.rename(lambda x: x.replace("_joint_prediction", ""))

            # which combos_df does row_nts match?
            match_combo = combos_df[(combos_df[nts] == row_nts.values).all(axis=1)]

            matching_nts = [nt for nt in nts if row_nts[nt]]

            # Convert matching_nts to a string
            matching_nts_str = ', '.join(matching_nts)

            # Is matching_nts_str already a key in cell_type_combos?
            if matching_nts_str not in cell_type_combos:
                cell_type_combos[matching_nts_str] = 1
            else:
                cell_type_combos[matching_nts_str] += 1

            number_of_nts = len(matching_nts)
            n_nts.append(number_of_nts)

            # Counts for nt_present
            for nt in matching_nts:
                nt_present[nt] += 1

        # convert nt_present into nt_percent
        nt_percent = {nt: (count / _df.shape[0]) * 100 for nt, count in nt_present.items()}
        
        test = {
            "classification": classification,
            "cell_type": cell_type,
            "combined_name": filter_name,
            "n_cells": _df.shape[0],
            "mean_n_nts": np.mean(n_nts),
            "std_n_nts": np.std(n_nts),
            "combos": cell_type_combos
        }
        for key in nt_percent:
            test[f"{key}_percent"] = nt_percent[key]
        all_cell_type_data.append(test)

cell_type_num_nts_df = pd.DataFrame(all_cell_type_data)
cell_type_num_nts_df.sort_values('cell_type', inplace=True)
cell_type_num_nts_df

#### Seaborn

In [ ]:
# Now a heat map for the NT_percent aspect
plt.figure(figsize=(12, 50))
sns.heatmap(
    cell_type_num_nts_df.set_index('combined_name')[['GABA_percent', 'ACh_percent', 'Glut_percent', 'Oct_percent', 'Ser_percent', 'DA_percent']],
    cmap="magma"
)
plt.show()
print(cell_type_num_nts_df['cell_type'].tolist())

In [ ]:
# Columns to visualise
selected_nt_cols = ["GABA_percent","ACh_percent","Glut_percent","Oct_percent",
           "Ser_percent","DA_percent","Unknown_percent"]

df_plot = (
    cell_type_num_nts_df
    .set_index("combined_name")[selected_nt_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
)

# Clustermap: rows & columns clustered; adjust metric/method as needed
g = sns.clustermap(
    data=df_plot,
    metric="euclidean",      # try "cosine" if you care about *pattern* more than magnitude
    method="average",        # linkage: "average", "ward", "complete", "single", etc.
    cmap="magma",          # perceptually uniform; "mako" or "rocket" are nice too
    col_cluster="false",
    dendrogram_ratio=(.15, 0.0),
    linewidths=0.0,
    figsize=(10, 25),
    cbar_pos=(1.02, .25, .02, .5),
) # move colorbar if you like


# Optional aesthetics
g.ax_heatmap.set_xlabel("Neurotransmitter")
g.ax_heatmap.set_ylabel("Cell type")

### Create a plotly version

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.figure_factory import create_dendrogram

# Columns to visualise
selected_nt_cols = ["GABA_percent","ACh_percent","Glut_percent","Oct_percent",
           "Ser_percent","DA_percent","Unknown_percent"]

df_plot = (
    cell_type_num_nts_df
    .set_index("combined_name")[selected_nt_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
)

# Row clustering
Z = linkage(df_plot.values, method="average", metric="euclidean")
row_leaves = leaves_list(Z)
row_order = df_plot.index[row_leaves]
M = df_plot.loc[row_order, :]  # keep original column order in df_plot

# Build a left (row) dendrogram
dendro = create_dendrogram(df_plot.values, 
                            orientation='right', 
                            labels=df_plot.index.tolist(),
                            linkagefun=lambda x: Z)

leaf_labels = list(dendro['layout']['yaxis']['ticktext'])
leaf_vals   = list(dendro['layout']['yaxis']['tickvals'])  # numeric positions used by the dendrogram
pos_by_label = dict(zip(leaf_labels, leaf_vals))

# Map heatmap rows to the dendrogram's y positions (order = M.index)
y_positions = [pos_by_label[label] for label in M.index]

# Subplots: 1 row x 2 cols (dendrogram left, heatmap right)
fig = make_subplots(rows=1, cols=2, column_widths=[0.1, 0.8],
                    specs=[[{'type':'heatmap'}, {'type':'xy'}]],
                    horizontal_spacing=0.2,
                    shared_yaxes=True)

# Add heatmap
fig.add_trace(
    go.Heatmap(
        z=M.values,
        x=M.columns,
        y=M.index,  # these are your real labels
        coloraxis="coloraxis",
        hovertemplate="Cell type: %{y}<br>NT: %{x}<br>Percent: %{z:.1f}%<extra></extra>"
    ),
    row=1, col=2
)

# Add dendrogram traces
for tr in dendro['data']:
    tr.showlegend = False 
    fig.add_trace(tr, row=1, col=1)

fig.update_traces(selector=dict(type='heatmap'), y=y_positions, row=1, col=2)

# Set identical y-ranges on both subplots
ymin, ymax = min(leaf_vals), max(leaf_vals)
pad = (ymax - ymin) * 0.002
fig.update_yaxes(range=[ymin - pad, ymax + pad], autorange=False, row=1, col=1)  # dendrogram
fig.update_yaxes(range=[ymin - pad, ymax + pad], autorange=False, row=1, col=2)  # heatmap

# Put readable ticks on the heatmap side only
fig.update_yaxes(
    tickmode="array",
    tickvals=y_positions,
    ticktext=M.index.tolist(),
    showticklabels=True,
    row=1, col=2
)
fig.update_yaxes(showticklabels=False, row=1, col=1)  # hide dendrogram ticks

# Axes & layout
fig.update_xaxes(matches=None, row=1, col=1, showticklabels=False)  # dendro x

fig.update_layout(
    width=1100, height=2500,
    margin=dict(l=20, r=60, t=40, b=40),
    coloraxis=dict(colorscale="Magma", colorbar=dict(title="Percent", x=1.02)),
)


fig.write_html("./fauai-16_data/clustermap_with_dendro_plotly.html", include_plotlyjs="cdn")
# fig.show()